In [17]:
import pandas as pd
import requests

In [18]:
import os
from dotenv import load_dotenv

# Load variables from .env into os.environ
env_path = r"C:\Users\bhave\OneDrive\Desktop\Machine Learning\.env"
load_dotenv(dotenv_path=env_path)   

# Access the keys
api_key = os.getenv("API_KEY")   

In [19]:
# response = requests.get(f'https://api.themoviedb.org/3/movie/top_rated?api_key={api_key}&language=en-US&page=1')
# response.json()

In [20]:
# pd.DataFrame(response.json()['results'])[['id','title','overview','release_date','popularity','vote_average','vote_count']]

In [21]:
df = pd.DataFrame()

In [ ]:
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Configure a retry strategy to automatically handle connection drops and rate limits
retry_strategy = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    raise_on_status=False
)

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry_strategy))
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
})

for i in range(1, 551):
    # session.get now automatically retries if the connection is reset or rate limited
    response = session.get(f'https://api.themoviedb.org/3/movie/top_rated?api_key={api_key}&language=en-US&page={i}')
    
    results = response.json().get('results')
    if not results:
        print(f"No results found on page {i}. Stopping.")
        break
        
    temp_df = pd.DataFrame(results)[['id','title','overview','release_date','popularity','vote_average','vote_count']]
    df = pd.concat([df, temp_df], ignore_index=True)
    
    # Small polite delay
    time.sleep(0.1)


No results found on page 501. Stopping.


In [24]:
df.shape

(10000, 7)

In [27]:
df.to_csv('Top_Rated_Movies.csv')